# 06 · Final comparison: GB, HistGB, CatBoost and ensembles
Stratified 80/20 split. The last cell searches ensemble weights on the same validation set it is scored on, so the 0.7042 is an optimistic number. The simple average (0.7036) is the fairer comparison.

In [8]:
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier
import numpy as np

# ---------------------------
# Load data
# ---------------------------
X_train = pd.read_csv("../Data/processed/X_train_clean.csv")
y_train = pd.read_csv("../Data/processed/y_train.csv")["overqualified"]

# ---------------------------
# Train/Validation split
# ---------------------------
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

# ---------------------------
# 1. Regular Gradient Boosting
# ---------------------------
gb = GradientBoostingClassifier(n_estimators=94, learning_rate=0.1, max_depth=3, random_state=42)
gb.fit(X_tr, y_tr)
pred_gb = gb.predict_proba(X_val)[:, 1]
score_gb = roc_auc_score(y_val, pred_gb)
print(f"GradientBoosting: {score_gb:.4f}")

# ---------------------------
# 2. HistGradientBoosting
# ---------------------------
hgb = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.01, max_depth=None, random_state=42)
hgb.fit(X_tr, y_tr)
pred_hgb = hgb.predict_proba(X_val)[:, 1]
score_hgb = roc_auc_score(y_val, pred_hgb)
print(f"HistGradientBoosting: {score_hgb:.4f}")

# ---------------------------
# 3. CatBoost
# ---------------------------
cat = CatBoostClassifier(iterations=200, learning_rate=0.05, random_state=42, verbose=0)
cat.fit(X_tr, y_tr)
pred_cat = cat.predict_proba(X_val)[:, 1]
score_cat = roc_auc_score(y_val, pred_cat)
print(f"CatBoost: {score_cat:.4f}")

# ---------------------------
# 4. Ensemble (simple average)
# ---------------------------
pred_ensemble = (pred_gb + pred_hgb + pred_cat) / 3
score_ensemble = roc_auc_score(y_val, pred_ensemble)
print(f"\nEnsemble (Average): {score_ensemble:.4f}")


GradientBoosting: 0.6999
HistGradientBoosting: 0.6932
CatBoost: 0.7011

Ensemble (Average): 0.7036


In [11]:
# Compute weights proportional to AUC
total_score = score_gb + score_hgb + score_cat
w_gb = score_gb / total_score
w_hgb = score_hgb / total_score
w_cat = score_cat / total_score

# Weighted ensemble
pred_ensemble = pred_gb * w_gb + pred_hgb * w_hgb + pred_cat * w_cat
score_ensemble = roc_auc_score(y_val, pred_ensemble)
print(f"\nEnsemble (Weighted): {score_ensemble:.4f}")



Ensemble (Weighted): 0.7036


In [12]:
best_score = 0
best_weights = None

# Generate weight grid (step 0.1)
for w_gb in np.arange(0, 1.1, 0.1):
    for w_hgb in np.arange(0, 1.1, 0.1):
        for w_cat in np.arange(0, 1.1, 0.1):
            if w_gb + w_hgb + w_cat == 0:
                continue
            # Normalize weights to sum to 1
            total = w_gb + w_hgb + w_cat
            wg, wh, wc = w_gb/total, w_hgb/total, w_cat/total
            ensemble = wg*pred_gb + wh*pred_hgb + wc*pred_cat
            score = roc_auc_score(y_val, ensemble)
            if score > best_score:
                best_score = score
                best_weights = (wg, wh, wc)

print(f"Best weights: GB={best_weights[0]:.2f}, HGB={best_weights[1]:.2f}, Cat={best_weights[2]:.2f}")
print(f"Best ensemble ROC-AUC: {best_score:.4f}")


Best weights: GB=0.37, HGB=0.11, Cat=0.53
Best ensemble ROC-AUC: 0.7042
